# Batch normalisation
The two steps by hand on a batch of 4, the same with Keras' layer, its parameters, training vs prediction,
and a comparison with and without batch normalisation on concentric circles.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
import numpy as np
import pandas as pd
import keras
import tensorflow as tf
from sklearn.datasets import make_circles

tf.config.experimental.enable_op_determinism()   # same numbers on every run
np.set_printoptions(precision=4, suppress=True)

## One node, one batch of 4: normalise, then scale and shift

In [2]:
z = np.array([2.0, 4.0, 6.0, 8.0])           # the node's weighted sums for the 4 rows of the batch
mu = z.mean()
var = z.var()                                # Keras divides by m, not m - 1
eps = 0.001                                  # Keras' default epsilon
z_norm = (z - mu) / np.sqrt(var + eps)
gamma, beta = 1.5, 0.5
z_bn = gamma * z_norm + beta
print("mean", mu, " variance", var, " std", np.sqrt(var).round(4))
print("normalised:", z_norm, " mean", z_norm.mean().round(6), " std", z_norm.std().round(4))
print("gamma = 1.5, beta = 0.5:", z_bn)

mean 5.0  variance 5.0  std 2.2361
normalised: [-1.3415 -0.4472  0.4472  1.3415]  mean 0.0  std 0.9999
gamma = 1.5, beta = 0.5: [-1.5123 -0.1708  1.1708  2.5123]


Keras' `BatchNormalization` layer, in training mode, with gamma and beta set to the same values.

In [3]:
bn = keras.layers.BatchNormalization()
bn.build((None, 1))
bn.gamma.assign([gamma]); bn.beta.assign([beta])
print(bn(z.reshape(-1, 1).astype("float32"), training=True).numpy().ravel())

[-1.5123 -0.1708  1.1708  2.5123]


Undoing the normalisation: gamma = sqrt(var + eps) and beta = mean give back the original z.

In [4]:
print(np.sqrt(var + eps) * z_norm + mu)

[2. 4. 6. 8.]


## The data: 500 points on two concentric circles

In [5]:
X, y = make_circles(n_samples=500, noise=0.1, factor=0.5, random_state=1)
X = ((X - X.mean(0)) / X.std(0)).astype("float32")
print(X.shape, np.bincount(y))

(500, 2) [250 250]


## Two models: without and with batch normalisation
Hidden layers of 3 and 2 ReLU nodes. The second model puts a `BatchNormalization` layer after each.

In [6]:
def build(bn, seed):
    keras.utils.set_random_seed(seed)
    layers = [keras.Input(shape=(2,)), keras.layers.Dense(3, activation="relu")]
    if bn:
        layers.append(keras.layers.BatchNormalization())
    layers.append(keras.layers.Dense(2, activation="relu"))
    if bn:
        layers.append(keras.layers.BatchNormalization())
    layers.append(keras.layers.Dense(1, activation="sigmoid"))
    m = keras.Sequential(layers)
    m.compile(optimizer=keras.optimizers.Adam(0.01), loss="binary_crossentropy", metrics=["accuracy"])
    return m

build(True, 0).summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 3)              │             9 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 3)              │            12 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │             8 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 2)              │             8 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             3 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 40 (160.00 B)

 Trainable params: 30 (120.00 B)

 Non-trainable params: 10 (40.00 B)

Each `BatchNormalization` layer stores 4 numbers per node: gamma, beta (trainable),
moving mean and moving variance (not trainable).

In [7]:
m = build(True, 0)
for w in m.layers[1].weights:
    print(f"{w.path:45s} shape {tuple(w.shape)}  trainable {w.trainable}")

sequential_1/batch_normalization_3/gamma      shape (3,)  trainable True
sequential_1/batch_normalization_3/beta       shape (3,)  trainable True
sequential_1/batch_normalization_3/moving_mean shape (3,)  trainable False
sequential_1/batch_normalization_3/moving_variance shape (3,)  trainable False


## Training both, 5 seeds each, 200 epochs, batch size 32
20% of the rows held out for validation.

In [8]:
curves, models = {}, {}
for bn in (False, True):
    for seed in range(5):
        m = build(bn, seed)
        h = m.fit(X, y, epochs=200, batch_size=32, validation_split=0.2, verbose=0).history
        curves[(bn, seed)] = h["val_accuracy"]
        models[(bn, seed)] = m
        print(f"batch norm {bn!s:5s} seed {seed}: val accuracy after 20 epochs {h['val_accuracy'][19]:.2f}, "
              f"after 200 {h['val_accuracy'][-1]:.2f}")
acc = pd.DataFrame({f"{'with' if bn else 'without'}_{seed}": v for (bn, seed), v in curves.items()})
acc.rename_axis("epoch").to_csv("data/val_accuracy.csv")
for k in ("without", "with"):
    mean = acc.filter(like=k + "_").mean(axis=1)
    first = int(np.argmax(mean.to_numpy() >= 0.8)) + 1 if (mean >= 0.8).any() else None
    print(f"{k:7s} batch norm: mean val accuracy epoch 20 {mean[19]:.2f}, epoch 200 {mean.iloc[-1]:.2f}, "
          f"first epoch with mean >= 0.80: {first}")

batch norm False seed 0: val accuracy after 20 epochs 0.76, after 200 0.79


batch norm False seed 1: val accuracy after 20 epochs 0.88, after 200 0.98


batch norm False seed 2: val accuracy after 20 epochs 0.80, after 200 0.98


batch norm False seed 3: val accuracy after 20 epochs 0.50, after 200 0.50


batch norm False seed 4: val accuracy after 20 epochs 0.96, after 200 0.98


batch norm True  seed 0: val accuracy after 20 epochs 0.90, after 200 0.92


batch norm True  seed 1: val accuracy after 20 epochs 0.77, after 200 0.99


batch norm True  seed 2: val accuracy after 20 epochs 0.71, after 200 0.98


batch norm True  seed 3: val accuracy after 20 epochs 0.97, after 200 0.98


batch norm True  seed 4: val accuracy after 20 epochs 0.83, after 200 0.96
without batch norm: mean val accuracy epoch 20 0.78, epoch 200 0.85, first epoch with mean >= 0.80: 23
with    batch norm: mean val accuracy epoch 20 0.84, epoch 200 0.97, first epoch with mean >= 0.80: 17


## What the second layer sees, epoch by epoch (data for the animations)
Seed 0 of both models again. After every epoch we store the 3 values that the first hidden layer passes on, for all
500 rows: the ReLU outputs without batch normalisation, and those outputs normalised with the 500 rows' own mean and
variance, then scaled by gamma and shifted by beta, with it. After every batch we store the first node's moving mean.

In [ ]:
class Recorder(keras.callbacks.Callback):
    def __init__(self, bn):
        super().__init__()
        self.bn, self.values, self.moving = bn, [], []
    def passed_on(self):
        a = self.model.layers[0](X).numpy()             # ReLU outputs of the first hidden layer
        if self.bn:                                      # normalised by hand: calling the layer would move its averages
            L = self.model.layers[1]
            a = L.gamma.numpy() * (a - a.mean(0)) / np.sqrt(a.var(0) + L.epsilon) + L.beta.numpy()
        return a
    def on_train_batch_end(self, batch, logs=None):
        if self.bn:
            self.moving.append(float(self.model.layers[1].moving_mean.numpy()[0]))
    def on_epoch_end(self, epoch, logs=None):
        self.values.append(self.passed_on())

rows, moving = [], {}
for bn in (False, True):
    m = build(bn, 0)
    rec = Recorder(bn)
    rec.set_model(m)
    rec.values.append(rec.passed_on())                  # epoch 0 = the start
    h = m.fit(X, y, epochs=200, batch_size=32, validation_split=0.2, verbose=0, callbacks=[rec]).history
    assert np.allclose(h["val_accuracy"], curves[(bn, 0)])   # recording does not change training
    for epoch, a in enumerate(rec.values):
        rows.append(pd.DataFrame(a, columns=["node1", "node2", "node3"]).assign(bn=bn, epoch=epoch))
    if bn:
        moving = pd.DataFrame(dict(step=np.arange(1, len(rec.moving) + 1), moving_mean=rec.moving))
passed = pd.concat(rows)
passed[passed.epoch % 4 == 0].round(3).to_csv("data/passed_on.csv.gz", index=False)   # every 4th epoch keeps data/ small
moving.round(5).to_csv("data/moving_mean.csv", index=False)
s = passed.groupby(["bn", "epoch"]).agg(["mean", "std"]).loc[(slice(None), [0, 20, 200]), :].round(2)
print(s.to_string())
print("moving mean of node 1 after the last batch:", round(moving.moving_mean.iloc[-1], 3))

## Training mode vs prediction mode
During training the layer uses the batch's own mean and variance. For prediction it uses the moving
averages it kept during training: moving = 0.99 x moving + 0.01 x batch value (Keras' momentum 0.99).

In [9]:
m_bn = models[(True, 0)]
bn_layer = m_bn.layers[1]
first = m_bn.layers[0](X).numpy()                    # outputs of the first hidden layer, all 500 rows
print("moving mean:     ", bn_layer.moving_mean.numpy())
print("actual mean:     ", first.mean(0))
print("moving variance: ", bn_layer.moving_variance.numpy())
print("actual variance: ", first.var(0))
one = X[:1]
print("one row, prediction mode:", m_bn(one, training=False).numpy().ravel(),
      "  same row in a batch of 1, training mode:", m_bn(one, training=True).numpy().ravel())

moving mean:      [0.2381 0.2219 1.4949]
actual mean:      [0.2255 0.2266 1.4845]
moving variance:  [0.1161 0.1075 0.5941]
actual variance:  [0.1196 0.1153 0.6197]
one row, prediction mode: [0.7856]   same row in a batch of 1, training mode: [0.5292]
